# Evaluation of Synthetic Data Generation Methods for Tabular Health Data

## Models Evaluated

In this study we evaluate both probabilistic and generative synthetic data generation methods on tabular health data,
using implementations from **Synthetic Data Vault (SDV)** and **synthcity** Python libraries.

For each model we also try a differentially private (DP) version and later evaluate both.

### Probabilistic

1. **Gaussian Copulas** - Baseline in general
    - `GaussianCopulaSynthesizer`
    - `DPGCSynthesizer` (DP version)
2. **Bayesian Networks**
    - `BayesianNetworkPlugin`
    - `PrivBayes` (DP version)

### Generative

1. **Generative Adversarial Networks**
    - `CTGAN` - baseline in generative models
    - DP variants: `DP-GAN`, `PATEGAN`, `AdsGAN`
2. **Variational Auto-Encoders**
    - `TVAE`, `RTVAE`
    - DP variants
3. **Normalizing Flows**
4. **Autoregressive / Tree-Based Models**
    - ARF (Adversarial Random Forests)
5. **Diffusion Models** (1)
    - `TabDDPM` (DP)
    - `TableDiffusion` (DP)

## Data

A public clinical dataset **Heart Disease** from **UC Irvine Machine Learning Repository** is used in this study.

In [ ]:
from ucimlrepo import fetch_ucirepo

heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

X

In [2]:
import pandas as pd

X = pd.read_csv("heart.csv")
X.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


# Generative Models

## Generative Adversarial Network (GAN)

In this section we use GAN models to generate synthetic datasets from our original data.

### General Idea

There are two components - generator and discriminator. Generator creates a realistic synthetic row of tabular data with random noise applied. Discriminator acts as a binary classifier deciding whether the provided row is real or fake (0-1), taking either real or synthetic row as an input.

`[insert GAN loss function here]`

Discriminator (D) wants to classify correctly real samples (high probability) and fake samples (low probability).

Generator (G) wants to make Discrimator (D) classify fake samples as real.

### GAN on Tabular Data

Conditional Tabular Generative Adversarial Network (CTGAN) can explicitly condition the data generation process on specific categorical variables.

A typical CTGAN training loop:

1. Sample real batch from dataset.
2. Sample noise z.
3. Generator creates synthetic samples G(z).
4. Discriminator trains on:
    - real samples → label 1
    - fake samples → label 0
5. Generator trains to fool the discriminator:
    - update G so D(G(z)) → 1
6. Apply conditional constraints (CTGAN)
7. Repeat until convergence.

In [2]:
from synthcity.plugins import Plugins

Plugins().list()

/workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


[2025-11-23T20:44:40.514295+0000][79342][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2025-11-23T20:44:40.518445+0000][79342][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py


['timevae',
 'adsgan',
 'radialgan',
 'survival_nflow',
 'decaf',
 'fflows',
 'ctgan',
 'dpgan',
 'ddpm',
 'uniform_sampler',
 'survival_gan',
 'aim',
 'privbayes',
 'tvae',
 'arf',
 'great',
 'survae',
 'marginal_distributions',
 'image_cgan',
 'image_adsgan',
 'dummy_sampler',
 'bayesian_network',
 'rtvae',
 'nflow',
 'pategan',
 'timegan',
 'survival_ctgan']

Create a train and test data loaders for **synthcity**.

In [3]:
from synthcity.plugins.core.dataloader import GenericDataLoader
loader = GenericDataLoader(
    X,
    target_column="target",
)
train_loader, test_loader = loader.train(), loader.test()

Load the CTGAN plugin class.

In [4]:
plugin_cls = type(Plugins().get("ctgan"))
plugin_cls
# plugin = Plugins().get("ctgan", n_iter = 100)

[2025-11-23T20:45:10.642427+0000][79342][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py


synthcity.plugins.generic.plugin_ctgan.CTGANPlugin

### CTGAN plugin in synthcity

Most parameters of the plugin describe architectures or training loop of Generator (G) and Discriminator (D) neural networks.

#### Generator parameters

- `generator_n_layers_hidden`: how many hidden layers the generator neural network has (excluding input/out). More layers -> more capacity, but slower + risk of overfitting.
- `generator_n_units_hidden`: number of neurons per hidden layer. Controls generator model size.
- `generator_nonlin`: activation function used between layers:
    - `leaky_relu` - default, stable for GANs
    - `relu` - simpler
    - `elu` / `selu` - sometimes help with continuous skewed data
- `generator_dropout`: fraction of neurons randomly dropped during training. Prevents overfitting. Usually 0 for tabular GANs.
- `n_iter`: total number of training iterations (epochs). Higher -> better model (until overfitting), longer training.

#### Discriminator parameters

- `discriminator_n_layers_hidden`: how many hidden layers the discriminator has.
- `discriminator_n_units_hidden`: number of neurons per layer
- `discriminator_nonlin`: activation function for the discriminator.
- `discriminator_dropout`: dropout regularization for the discriminator.
- `discriminator_n_inter`: number of discriminator update steps per cycle.

#### Optimization and training

- `lr`: learning rate for both generator and discriminator
- `weight_decay`: L2 regularization on weights, which prevents exploding weights (and overfitting). Adds penalty to the loss function proportional to the sum of squares of the model's weights.
- `batch_size`: mini-batch size used during training. For a small dataset like in our case (303 rows), values 32-128 work good.
- `random_state`: random seed for reproducibility.
- `clipping_value`: gradient clipping threshold.

#### Preprocessing / encoding

- `encoder_max_clusters`: continuous variables are encoded using mode-specific normalization. This parameter controls the max number of mixture components (clusters) used. More clusters = better modeling of multimodal distributions, slower training.

To automatically tune hyperparameters in `synthcity`, we use hyperparameter optimization (HPO) algorithms such as Tree-structured Parzen estimators (TPE), Bayesian optimization, and genetic programming.

We will use `optuna` HPO library, which implements TPE, to tune hyperparameters.

Display the hyperparameter space.

In [5]:
plugin_cls.hyperparameter_space()

[IntegerDistribution(name='generator_n_layers_hidden', data=None, random_state=None, sampling_strategy='marginal', marginal_distribution=None, low=1, high=4, step=1),
 IntegerDistribution(name='generator_n_units_hidden', data=None, random_state=None, sampling_strategy='marginal', marginal_distribution=None, low=50, high=150, step=50),
 CategoricalDistribution(name='generator_nonlin', data=None, random_state=None, sampling_strategy='marginal', marginal_distribution=elu           0.25
 leaky_relu    0.25
 relu          0.25
 tanh          0.25
 dtype: float64, choices=['elu', 'leaky_relu', 'relu', 'tanh']),
 IntegerDistribution(name='n_iter', data=None, random_state=None, sampling_strategy='marginal', marginal_distribution=None, low=100, high=1000, step=100),
 FloatDistribution(name='generator_dropout', data=None, random_state=None, sampling_strategy='marginal', marginal_distribution=None, low=0.0, high=0.2),
 IntegerDistribution(name='discriminator_n_layers_hidden', data=None, random_st

Use a trial to suggest a set of hyperparameters.

In [6]:
import optuna
from synthcity.utils.optuna_sample import suggest_all

trial = optuna.create_study().ask()
params = suggest_all(trial, plugin_cls.hyperparameter_space())
# params["n_iter"] = 100 # speed up
params

{'generator_n_layers_hidden': 1,
 'generator_n_units_hidden': 100,
 'generator_nonlin': 'relu',
 'n_iter': 200,
 'generator_dropout': 0.16423752147137477,
 'discriminator_n_layers_hidden': 3,
 'discriminator_n_units_hidden': 100,
 'discriminator_nonlin': 'elu',
 'discriminator_n_iter': 5,
 'discriminator_dropout': 0.039818031175037905,
 'lr': 0.0002,
 'weight_decay': 0.0001,
 'batch_size': 500,
 'encoder_max_clusters': 6}

Evaluate the plugin with the suggested hyperparameters.

In [17]:
from synthcity.benchmark import Benchmarks

plugin = plugin_cls(**params).fit(train_loader)
report = Benchmarks.evaluate(
    [("trial", "ctgan", params)],
    train_loader, # Benchmarks.evaluate will split out a validation set
    repeats=1,
    # metrics={"detection": ["detection_xgb"]}
)
report["trial"]

100%|██████████| 200/200 [00:12<00:00, 16.46it/s]
[2025-11-23T18:14:44.208254+0000][3798][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
100%|██████████| 200/200 [00:10<00:00, 18.98it/s]


,min,max,mean,stddev,median,iqr,rounds,errors,durations,direction
sanity.data_mismatch.score,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,0.00,minimize
sanity.common_rows_proportion.score,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,0.00,minimize
sanity.nearest_syn_neighbor_distance.mean,0.258148,0.258148,0.258148,0.0,0.258148,0.0,1,0,0.00,minimize
sanity.close_values_probability.score,0.439024,0.439024,0.439024,0.0,0.439024,0.0,1,0,0.00,maximize
sanity.distant_values_probability.score,0.006098,0.006098,0.006098,0.0,0.006098,0.0,1,0,0.00,minimize
stats.jensenshannon_dist.marginal,0.044006,0.044006,0.044006,0.0,0.044006,0.0,1,0,0.06,minimize
stats.chi_squared_test.marginal,0.627289,0.627289,0.627289,0.0,0.627289,0.0,1,0,0.02,maximize
stats.inv_kl_divergence.marginal,0.696748,0.696748,0.696748,0.0,0.696748,0.0,1,0,0.01,maximize
stats.ks_test.marginal,0.643728,0.643728,0.643728,0.0,0.643728,0.0,1,0,0.01,maximize
stats.max_mean_discrepancy.joint,0.014872,0.014872,0.014872,0.0,0.014872,0.0,1,0,0.01,minimize


Create an Optuna study and optimize the hyperparameters.

In [7]:
from synthcity.benchmark import Benchmarks

def objective(trial: optuna.Trial):
    hp_space = Plugins().get("ctgan").hyperparameter_space()
    params = suggest_all(trial, hp_space)
    ID = f"trial_{trial.number}"
    try:
        report = Benchmarks.evaluate(
            [(ID, "ctgan", params)],
            train_loader,
            repeats=1,
            metrics={"stats": ["jensenshannon_dist"]},
        )
    except Exception:
        raise optuna.TrialPruned()

    score = report[ID]["mean"].mean()
    return score

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)
best_params = study.best_params

[2025-11-23T21:03:08.661980+0000][79342][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2025-11-23T21:03:08.701050+0000][79342][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
 90%|████████▉ | 449/500 [00:59<00:06,  7.54it/s]
[2025-11-23T21:04:10.078038+0000][79342][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2025-11-23T21:04:10.086046+0000][79342][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
 50%|████▉     | 499/1000 [01:11<01:11,  6.99it/s]
[2025-11-23T21:05:22.743922+0000][79342][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py

In [ ]:
# import pandas as pd

# def objective(trial: optuna.Trial):
#     hp_space = Plugins().get("ctgan").hyperparameter_space()
#     params = suggest_all(trial, hp_space)
#     ID = f"trial_{trial.number}"
#     try:
#         report = Benchmarks.evaluate(
#             [(ID, "ctgan", params)],
#             train_loader,
#             repeats=1,
#             metrics={"detection": ["detection_xgb"]},
#         )
#     except Exception as e: # invalid set of params
#         print(f"{type(e).__name__}: {e}")
#         print(params)
#         raise optuna.TrialPruned()
#     score = report[ID].query('direction == "minimize"')["mean"].mean()
#     # df = report[ID]
#     # minimize metrics
#     # minimize_vals = df.query('direction == "minimize"')["mean"]
#     # maximize metrics → turn into minimize form by negating
#     # maximize_vals = -df.query('direction == "maximize"')["mean"]
#     # final scalar objective
#     # score = pd.concat([minimize_vals, maximize_vals]).mean()

#     return score

# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=2)
# study.best_params

[2025-11-23T19:12:12.612630+0000][3798][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2025-11-23T19:12:12.738518+0000][3798][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
100%|██████████| 200/200 [01:13<00:00,  2.73it/s]
[2025-11-23T19:13:28.214185+0000][3798][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2025-11-23T19:13:28.223149+0000][3798][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
100%|██████████| 500/500 [01:23<00:00,  6.01it/s]


{'generator_n_layers_hidden': 2,
 'generator_n_units_hidden': 150,
 'generator_nonlin': 'relu',
 'n_iter': 500,
 'generator_dropout': 0.13266420537712784,
 'discriminator_n_layers_hidden': 2,
 'discriminator_n_units_hidden': 150,
 'discriminator_nonlin': 'tanh',
 'discriminator_n_iter': 3,
 'discriminator_dropout': 0.08724841559085626,
 'lr': 0.0002,
 'weight_decay': 0.001,
 'batch_size': 200,
 'encoder_max_clusters': 18}

Visualize the study.

In [ ]:
from optuna.visualization import (plot_contour,
                                  plot_edf,
                                  plot_optimization_history,
                                  plot_parallel_coordinate,
                                  plot_param_importances,
                                  plot_slice)

plot_optimization_history(study)

In [ ]:
# visualize high-dimensional parameter relationships
plot_parallel_coordinate(study)

In [ ]:
# visualize hyperparameter relationships
fig = plot_contour(study, params=["batch_size", "lr", "generator_n_layers_hidden", "discriminator_n_layers_hidden"])
fig.update_layout(width=800, height=800)

In [ ]:
# visualize individual hyperparameters as slice plot
plot_slice(study)

In [ ]:
# visualize parameter importances
plot_param_importances(study)

In [ ]:
# learn which hyperparameters are affecting the trial duration with hyperparameter importance
optuna.visualization.plot_param_importances(
    study, target=lambda t: t.duration.total_seconds(), target_name="duration"
)

In [ ]:
# visualize empirical distribution function of the objective
plot_edf(study)

Test performance of the optimized plugin.

In [8]:
best_params = study.best_params
report = Benchmarks.evaluate(
    [("test", "ctgan", best_params)],
    train_loader,
    test_loader,
    repeats=1,
    # metrics={"detection": ["detection_mlp", "detection_xgb"]},
)
Benchmarks.print(report)

[2025-11-23T21:08:28.999305+0000][79342][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
100%|██████████| 800/800 [00:53<00:00, 15.01it/s]



Plugin : test


,min,max,mean,stddev,median,iqr,rounds,errors,durations
sanity.data_mismatch.score,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,0.00
sanity.common_rows_proportion.score,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,0.01
sanity.nearest_syn_neighbor_distance.mean,0.058598,0.058598,0.058598,0.0,0.058598,0.0,1,0,0.01
sanity.close_values_probability.score,0.985366,0.985366,0.985366,0.0,0.985366,0.0,1,0,0.00
sanity.distant_values_probability.score,0.004878,0.004878,0.004878,0.0,0.004878,0.0,1,0,0.00
stats.jensenshannon_dist.marginal,0.010985,0.010985,0.010985,0.0,0.010985,0.0,1,0,0.09
stats.chi_squared_test.marginal,0.629321,0.629321,0.629321,0.0,0.629321,0.0,1,0,0.03
stats.inv_kl_divergence.marginal,0.901631,0.901631,0.901631,0.0,0.901631,0.0,1,0,0.02
stats.ks_test.marginal,0.920906,0.920906,0.920906,0.0,0.920906,0.0,1,0,0.03
stats.max_mean_discrepancy.joint,0.011993,0.011993,0.011993,0.0,0.011993,0.0,1,0,0.05


In [20]:
best_params

{'generator_n_layers_hidden': 2,
 'generator_n_units_hidden': 150,
 'generator_nonlin': 'relu',
 'n_iter': 500,
 'generator_dropout': 0.13266420537712784,
 'discriminator_n_layers_hidden': 2,
 'discriminator_n_units_hidden': 150,
 'discriminator_nonlin': 'tanh',
 'discriminator_n_iter': 3,
 'discriminator_dropout': 0.08724841559085626,
 'lr': 0.0002,
 'weight_decay': 0.001,
 'batch_size': 200,
 'encoder_max_clusters': 18}

In [21]:
plugin.generate(50, **best_params)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,51,1,2,112,473,0,2,192,0,4.301238,0,2,2,0
1,62,1,2,109,298,0,1,202,0,2.113944,1,1,2,0
2,52,0,0,117,206,0,1,199,1,6.200000,2,2,3,0
3,47,1,2,118,400,0,2,155,1,0.000000,2,1,2,0
4,60,0,2,131,207,1,0,174,0,1.905392,1,0,3,1
5,62,1,2,124,285,0,0,173,1,0.850394,2,2,2,0
6,65,0,3,113,564,0,1,194,0,2.904682,2,2,3,0
7,49,0,3,104,330,0,1,202,1,0.760005,1,1,2,1
8,47,0,2,104,312,0,1,187,1,2.750711,1,3,3,0
9,50,0,0,129,253,1,2,146,0,2.059770,1,0,1,0


Metrics we will use in the study.

1. Fidelity:
    - Jensen-Shannon distance (marginal)
    - Wasserstein distance (joint)
    - PRDC (precision/recall)

2. Utility
    - Train on synthetic -> test on real (TSTR) performance. Using a consistent ML model (e.g., XGBoost or Logistic Regression)

3. Privacy
    - Nearest-neighbor distance
    - Common rows proportion
    - Membership inference attach AUC
    - etc.

In [ ]:
from src.run_experiment import run_experiment

run_experiment("rtvae", "heart.csv")

/workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.
=== Optimizing hyperparameters ===


[2025-11-23T23:01:43.713814+0000][137330][CRITICAL] Error importing TabularGoggle: No module named 'dgl'
[2025-11-23T23:01:43.718043+0000][137330][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2025-11-23T23:01:45.190043+0000][137330][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
100%|██████████| 400/400 [01:02<00:00,  6.43it/s]
[2025-11-23T23:02:48.740125+0000][137330][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
[2025-11-23T23:02:48.748522+0000][137330][CRITICAL] module disabled: /workspaces/synth_tab_health_data/.venv/lib/python3.12/site-packages/synthcity/plugins/generic/plugin_goggle.py
100%|██████████| 200/200 [01:11<00:00,  2.80it/s]
[2025-11-23T23:04:01.951706+0000][137330][CRITICAL] module disabled: /w


=== Best parameters ===
{'n_iter': 200, 'lr': 0.001, 'decoder_n_layers_hidden': 4, 'weight_decay': 0.001, 'batch_size': 512, 'n_units_embedding': 450, 'decoder_n_units_hidden': 50, 'decoder_nonlin': 'elu', 'decoder_dropout': 0.1402374823431258, 'encoder_n_layers_hidden': 1, 'encoder_n_units_hidden': 400, 'encoder_nonlin': 'elu', 'encoder_dropout': 0.12412264944660674}

=== Evaluating all metrics with best parameters ===


100%|██████████| 200/200 [01:08<00:00,  2.94it/s]



Plugin : test


,min,max,mean,stddev,median,iqr,rounds,errors,durations
sanity.data_mismatch.score,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,0.00
sanity.common_rows_proportion.score,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,0.01
sanity.nearest_syn_neighbor_distance.mean,0.098060,0.098060,0.098060,0.0,0.098060,0.0,1,0,0.01
sanity.close_values_probability.score,0.926829,0.926829,0.926829,0.0,0.926829,0.0,1,0,0.00
sanity.distant_values_probability.score,0.004878,0.004878,0.004878,0.0,0.004878,0.0,1,0,0.00
stats.jensenshannon_dist.marginal,0.022204,0.022204,0.022204,0.0,0.022204,0.0,1,0,0.06
stats.chi_squared_test.marginal,0.462032,0.462032,0.462032,0.0,0.462032,0.0,1,0,0.02
stats.inv_kl_divergence.marginal,0.686190,0.686190,0.686190,0.0,0.686190,0.0,1,0,0.01
stats.ks_test.marginal,0.841812,0.841812,0.841812,0.0,0.841812,0.0,1,0,0.01
stats.max_mean_discrepancy.joint,0.012886,0.012886,0.012886,0.0,0.012886,0.0,1,0,0.02


Describe models in chronological order, timeline.